## Tennis Versus (05 - adding ELO)

Added more data (now 2009 - 2026), Filtered out burn-in rows (like the early years).

you know what, maybe adding more data skewed the data because the game's changed.

It's time to add an ELO feature.

In [4]:
# import Logistic Regression from sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

In [5]:
# data analytics + math
import pandas as pd 
import numpy as np 

# data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# other stuff used for later (need sci-kit learn too)
import xgboost # just installed it using !pip install

# set some parameters for pandas to view the data a bit better
pd.set_option('display.max_columns', 50) # see ALL columns
pd.set_option('display.width', 150)

data = pd.read_parquet('restructured_matches_02done.parquet')

In [6]:
# all the 'dependent' variables
features = data.columns.tolist()
features.remove('label')
features.remove('tourney_date')

X = data[features]
y = data['label']

print(X.tail())

        age_a   age_b  rank_diff  form_a  form_b  surface_form_a  surface_form_b  h2h_winrate_a        elo_a        elo_b  surface_elo_a  \
66725  29.092  27.984       -5.0     0.8     0.7             0.8             0.8       1.000000  1943.483455  1773.523780    1765.357452   
66726  28.646  28.416      -17.0     0.7     0.8             0.8             0.8       0.875000  1910.118958  1913.270619    1919.225866   
66727  27.855  29.095       -1.0     0.7     0.8             0.4             0.4       0.714286  1866.212842  1952.226367    1601.757161   
66728  28.416  28.646       17.0     0.8     0.7             0.7             0.7       0.125000  1929.125483  1894.264094    1735.003566   
66729  29.095  27.855        1.0     0.8     0.7             0.8             0.8       0.285714  1932.344286  1886.094924    1752.116427   

       surface_elo_b    elo_diff  surface_elo_diff  bp_pressure_a  bp_pressure_b  surface_Clay  surface_Grass  surface_Hard  
66725    1632.268002  169.959674 

### training and testing

Our data is from 2014 - current, so let's train on 2016 - 2024. Let's use 2025 - current as testing set.

In [7]:
train = data[
    (data['tourney_date'] >= '2016-01-01') &   # skip 2009 and 2010 as burn-in
    (data['tourney_date'] < '2025-01-01')
]

test = data[
    data['tourney_date'] >= '2025-01-01'
]

X_train = train[features]
y_train = train['label']

X_test = test[features]
y_test = test['label']

# verifying shape
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(47910, 19)
(47910,)
(8341, 19)
(8341,)


### scaling

again, using StandardScaler()

In [8]:
scaler = StandardScaler()

# scale it --> turn it into numpy array
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# save the scaler for late?
# pickle: object serializer / deserializer -- used to save objects in Python throughout files
# import pickle 
# pickle.dump(scaler, open('scaler2.pkl', 'wb'))

In [10]:
X_train_scaled

array([[-1.09849387, -0.01917953,  0.03220683, ..., -0.66009144,
        -0.34802493,  0.83624023],
       [-0.39850457, -0.19472407, -0.02760585, ..., -0.66009144,
        -0.34802493,  0.83624023],
       [-0.36304545,  0.70729467,  0.00460098, ..., -0.66009144,
        -0.34802493,  0.83624023],
       ...,
       [ 0.41661744,  0.45601646,  0.04600976, ..., -0.66009144,
        -0.34802493,  0.83624023],
       [-0.04019234,  0.11521491, -0.05521171, ..., -0.66009144,
        -0.34802493,  0.83624023],
       [-0.6878685 ,  0.11521491,  0.04140878, ..., -0.66009144,
        -0.34802493,  0.83624023]], shape=(47910, 19))

### model training once again

For now, I will stick with ```LogisticRegression()```.

In [13]:
model = LogisticRegression(max_iter=1000) # set 1000 for now ( default is 100 and sometimes thats not enough for larger datasets)
model.fit(X_train_scaled, y_train)

model.coef_ 
# ^ note: surface_elo diff and elo_diff is 6th and 7th from the bottom
# their coeffs are very high !

array([[-0.11785477,  0.10508109, -0.19979502, -0.22039393,  0.39739094,
        -0.16081507, -0.16081507, -0.00470912,  0.20393947, -0.41133656,
         0.42578511, -0.1000666 ,  0.5078369 ,  0.43368461, -0.03373505,
         0.02761637,  0.0076856 ,  0.047864  , -0.03737335]])

In [17]:
# predictions
y_pred = model.predict(X_test_scaled) # will give a hard 0 or 1
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1] # will give a probability instead ( returns [failing, passing] we want index 1)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
logloss = log_loss(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Log loss:  {logloss:.3f}")
print(f"AUC-ROC:   {auc:.3f}")

naive = (X_test['rank_diff'] < 0).astype(int)
naive_accuracy = accuracy_score(y_test, naive)

print(f"Naive accuracy:  {naive_accuracy:.3f}")

Accuracy:  0.689
Log loss:  0.583
AUC-ROC:   0.759
Naive accuracy:  0.645


### Conclusion

We finally beat native accuracy!

Logistic Regression (with ELO): 68.9%

Just pick the higher ranked guy: 64.5%

In [20]:
# actually let's save the data
data.to_parquet('elo_data.parquet')